# CodeAlpha Machine Learning Internship
## Task 3: Handwritten Character Recognition

**Objective**: Identify handwritten characters/digits from image data using Convolutional Neural Networks (CNN).
- **Dataset**: MNIST Benchmark Dataset
- **Model Architecture**: Deep Convolutional Neural Network with Batch Normalization and Dropout
- **Evaluation Metrics**: Accuracy, Loss Curves, Classification Report, Confusion Matrix

In [ ]:
# 1. Import Essential Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using compute device: {device}")

### 2. Dataset Loading and Preprocessing
We apply transformations including normalization and data augmentation to improve generalization.

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform_train)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"Training samples: {len(train_dataset)} | Testing samples: {len(test_dataset)}")

### 3. CNN Model Architecture
Constructing a multi-layer Convolutional Neural Network with MaxPooling, Batch Normalization, and Dropout.

In [ ]:
class CharacterCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CharacterCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 28x28 -> 14x14

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 14x14 -> 7x7

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)   # 7x7 -> 3x3
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CharacterCNN(num_classes=10).to(device)
print(model)

### 4. Training the Convolutional Neural Network

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
epochs = 3

for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * data.size(0)
        _, predicted = outputs.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()

    acc = 100. * correct / total
    print(f"Epoch [{epoch}/{epochs}] - Loss: {running_loss/total:.4f} | Training Accuracy: {acc:.2f}%")

### 5. Model Evaluation and Metrics Report

In [ ]:
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        outputs = model(data)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(target.cpu().numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
test_acc = 100.0 * np.sum(all_preds == all_targets) / len(all_targets)

print(f"🎯 Final Test Accuracy: {test_acc:.2f}%")
print("
📊 Classification Report:
")
print(classification_report(all_targets, all_preds, digits=4))

### 6. Visualizing Sample Predictions

In [ ]:
test_batch, label_batch = next(iter(test_loader))
model.eval()
with torch.no_grad():
    sample_outputs = model(test_batch[:8].to(device))
    _, sample_preds = torch.max(sample_outputs, 1)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for idx, ax in enumerate(axes.flat):
    img = test_batch[idx].squeeze().numpy()
    ax.imshow(img, cmap="gray")
    true_label = label_batch[idx].item()
    pred_label = sample_preds[idx].item()
    color = "green" if true_label == pred_label else "red"
    ax.set_title(f"True: {true_label} | Pred: {pred_label}", color=color, fontsize=11, fontweight="bold")
    ax.axis("off")
plt.tight_layout()
plt.show()